In [43]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("gold_business_metrics").getOrCreate()

In [44]:
# Reading the data from parquet files from Silver layer to do gold aggregations

customers_silver = spark.read.parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/silver/customers")
orders_silver = spark.read.parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/silver/orders")
order_items_silver = spark.read.parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/silver/order_items")
payments_silver = spark.read.parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/silver/payments")

In [45]:
customers_silver.show(3)
orders_silver.show(3)
order_items_silver.show(3)
payments_silver.show(3)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|f2a1d75b74d9ec748...|15ee900ec703c9a10...|                   68590|             Jacunda|            PA|
|f15272fe9d0e2ae32...|11e74a9cbe1158d1c...|                   15056|Sao Jose Do Rio P...|            SP|
|7324ecb0ff143f561...|c6be127fa6e30c6f7...|                   13302|                 Itu|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 3 rows
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------------------+-------------+
|           

In [46]:
# Aggregated the Order items into corresponding orders, so that each row is one order

from pyspark.sql.functions import sum, count, round
order_items_agg = order_items_silver.groupBy("order_id").agg(round((sum("total_item_value")),2).alias("total_price")\
                                                            ,count("order_item_id").alias("total_items_per_order")
                                                            ).orderBy("total_items_per_order", ascending = False)
order_items_agg.show()

[Stage 264:==============>                                          (1 + 3) / 4]

+--------------------+-----------+---------------------+
|            order_id|total_price|total_items_per_order|
+--------------------+-----------+---------------------+
|8272b63d03f5f79c5...|     196.17|                   21|
|ab14fdcfbe524636d...|     2262.8|                   20|
|1b15974a0141d54e3...|     2202.4|                   20|
|428a2f660dc84138d...|    1225.65|                   15|
|9ef13efd6949e4573...|      783.0|                   15|
|73c8ab38f07dc9438...|    1014.02|                   14|
|9bdc4d4c71aa1de46...|     528.78|                   14|
|37ee401157a3a0b28...|     485.94|                   13|
|c05d6a79e55da72ca...|     302.52|                   12|
|af822dacd6f5cff73...|     244.02|                   12|
|637617b3ffe9e2f7a...|    1246.95|                   12|
|2c2a19b5703863c90...|     441.72|                   12|
|3a213fcdfe7d98be7...|    1482.24|                   12|
|6c355e2913545fa6f...|     540.76|                   11|
|5a3b1c29a49756e75...|     853.

In [47]:
# Aggregated multiple payments into single order payment, so that each row is one order and total amount paid by the customer for that order

payments_agg = payments_silver.groupBy("order_id").agg(\
                                                        round((sum("payment_value")),2).alias("total_payment_value"),\
                                                        sum("payment_installments").alias("total_installments")
                                                        )
payments_agg.show()

[Stage 267:>                                                        (0 + 4) / 4]

+--------------------+-------------------+------------------+
|            order_id|total_payment_value|total_installments|
+--------------------+-------------------+------------------+
|50f565fbb2600be9e...|             101.46|                 5|
|26ebbef3221e8b51c...|              94.21|                 4|
|416699ef2e608b6f2...|             146.02|                 4|
|a92225110bfd8206a...|              67.23|                 3|
|0c077fbe69b84abe1...|              70.89|                 2|
|6f64e3dc6ee08865d...|               50.1|                 1|
|a6ade711f329afba0...|              96.65|                 3|
|69340678ddc4f528c...|             142.98|                 6|
|76f9204ee569a062f...|             198.99|                 2|
|e5ede2994d0d9af52...|              68.64|                 2|
|236d48ee787db9dbd...|               51.1|                 1|
|e6485b91715e7f242...|             143.39|                 1|
|5137d2621656ff029...|             133.34|                 1|
|7a5472f

In [48]:
# Joining Orders df, Customers df, Order items & Payments, so that each row is one order

gold_orders = orders_silver.join(customers_silver, "customer_id", "inner")\
    .join(order_items_agg, "order_id", "left")\
    .join(payments_agg, "order_id", "left")
gold_orders.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------------------+-------------+--------------------+------------------------+-------------------+--------------+-----------+---------------------+-------------------+------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|approval_delivery_valid|delivery_days|  customer_unique_id|customer_zip_code_prefix|      customer_city|customer_state|total_price|total_items_per_order|total_payment_value|total_installments|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------------------+-------------+------

In [49]:
gold_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- approval_delivery_valid: string (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- total_price: double (nullable = true)
 |-- total_items_per_order: long (nullable = true)
 |-- total_payment_value: double (nullable = true)
 |-- total_installments: long (nullable = true)



In [50]:
# Selecting few columns and ignored few which are not necessary

gold_orders = gold_orders.select("order_id", "total_price", "total_payment_value", "total_items_per_order", "customer_id", "customer_city",\
                                 "customer_state",\
                                 "customer_zip_code_prefix", "order_status",\
                                 "order_purchase_timestamp", "order_delivered_customer_date", "approval_delivery_valid", "delivery_days",\
                                 "total_installments")
gold_orders.show(5)

+--------------------+-----------+-------------------+---------------------+--------------------+-------------------+--------------+------------------------+------------+------------------------+-----------------------------+-----------------------+-------------+------------------+
|            order_id|total_price|total_payment_value|total_items_per_order|         customer_id|      customer_city|customer_state|customer_zip_code_prefix|order_status|order_purchase_timestamp|order_delivered_customer_date|approval_delivery_valid|delivery_days|total_installments|
+--------------------+-----------+-------------------+---------------------+--------------------+-------------------+--------------+------------------------+------------+------------------------+-----------------------------+-----------------------+-------------+------------------+
|6ec1bea8cbcef0a1b...|      36.32|              36.32|                    1|e6b5e20566e5c72cb...|Sao Jose De Ribamar|            MA|                   

In [51]:
# Checking if there are any duplicate orders

gold_orders.groupBy("order_id").count().filter("count >1").show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [52]:
# Deriving a new column to flag, if there is any difference between the total price of the item, and the amount paid towards that item
# It's a data quality check column
# Also deriving a new column to calculate the difference amount
# This becomes the main Fact Table

from pyspark.sql.functions import when, col
gold_orders = gold_orders.withColumn("payment_mismatch_flag", when((col("total_price") != col("total_payment_value")), "Yes")
                         .otherwise("No")
                      ).withColumn("payment_difference", col("total_price")- col("total_payment_value"))
                             
gold_orders.show(5)

+--------------------+-----------+-------------------+---------------------+--------------------+-------------------+--------------+------------------------+------------+------------------------+-----------------------------+-----------------------+-------------+------------------+---------------------+------------------+
|            order_id|total_price|total_payment_value|total_items_per_order|         customer_id|      customer_city|customer_state|customer_zip_code_prefix|order_status|order_purchase_timestamp|order_delivered_customer_date|approval_delivery_valid|delivery_days|total_installments|payment_mismatch_flag|payment_difference|
+--------------------+-----------+-------------------+---------------------+--------------------+-------------------+--------------+------------------------+------------+------------------------+-----------------------------+-----------------------+-------------+------------------+---------------------+------------------+
|6ec1bea8cbcef0a1b...|      

In [53]:
# Customer Analytics, which customers has more no.of orders, total price of all orders, avg price. 
# But unfortunately this dataset has 1 order per customer

from pyspark.sql.functions import avg

gold_customer_metrics = gold_orders.groupBy("customer_id").agg(count("order_id").alias("total_orders"),
                                                               sum("total_price").alias("total_spent"),
                                                               avg("total_price").alias("avg_order_value"))\
                                                          .orderBy(col("total_spent"), ascending = False)
gold_customer_metrics.show(5)

+--------------------+------------+-----------+---------------+
|         customer_id|total_orders|total_spent|avg_order_value|
+--------------------+------------+-----------+---------------+
|1617b1357756262bf...|           1|   13664.08|       13664.08|
|ec5b2ba62e5743423...|           1|    7274.88|        7274.88|
|c6e2731c5b391845f...|           1|    6929.31|        6929.31|
|f48d464a0baaea338...|           1|    6922.21|        6922.21|
|3fd6777bbce08a352...|           1|    6726.66|        6726.66|
+--------------------+------------+-----------+---------------+
only showing top 5 rows


In [54]:
# State Level Sales

gold_sale_by_states = gold_orders.groupBy("customer_state").agg(count("order_id").alias("total_orders"),
                                                                round((sum("total_price")),2).alias("total_revenue"),
                                                                round((avg("delivery_days")),1).alias("avg_delivery_days")
                                                               )\
                                                            .orderBy(("total_orders"),("total_revenue"),ascending = False)
gold_sale_by_states.show()

+--------------+------------+-------------+-----------------+
|customer_state|total_orders|total_revenue|avg_delivery_days|
+--------------+------------+-------------+-----------------+
|            SP|       41746|   5921678.12|              8.7|
|            RJ|       12852|   2129681.98|             15.2|
|            MG|       11635|   1856161.49|             11.9|
|            RS|        5466|    885826.76|             15.2|
|            PR|        5045|    800935.44|             11.9|
|            SC|        3637|     610213.6|             14.9|
|            BA|        3380|    611506.67|             19.3|
|            DF|        2140|    353229.44|             12.9|
|            ES|        2033|    324801.91|             15.7|
|            GO|        2020|    347706.93|             15.5|
|            PE|        1652|    322237.69|             18.4|
|            CE|        1336|     275606.3|             21.2|
|            PA|         975|    217647.11|             23.7|
|       

In [55]:
# Daily Sales Trend

gold_daily_sales = gold_orders. groupBy("order_purchase_timestamp").agg(count("order_id").alias("total_orders"),
                                                                       round((sum("total_price")),2).alias("daily_revenue"))\
                                                                    .orderBy("total_orders", ascending = False)
gold_daily_sales.show()

+------------------------+------------+-------------+
|order_purchase_timestamp|total_orders|daily_revenue|
+------------------------+------------+-------------+
|     2017-11-20 11:46:30|           3|       867.81|
|     2018-02-19 15:37:47|           3|       338.46|
|     2018-08-02 12:06:07|           3|       512.57|
|     2017-11-20 10:59:08|           3|       164.41|
|     2018-06-01 13:39:44|           3|       127.65|
|     2018-08-02 12:06:09|           3|      1262.44|
|     2018-04-11 10:48:14|           3|       1232.4|
|     2018-08-02 12:05:26|           3|       434.58|
|     2018-03-31 15:08:21|           3|       177.96|
|     2018-07-28 13:11:22|           3|       862.41|
|     2017-06-07 14:10:15|           2|        116.9|
|     2017-08-31 10:28:39|           2|        39.34|
|     2017-10-31 22:19:42|           2|       420.48|
|     2017-05-10 09:00:07|           2|       752.68|
|     2018-05-14 11:29:01|           2|       193.66|
|     2018-06-12 18:14:33|  

In [56]:
from pyspark.sql.functions import min, max

gold_orders.select(
    min("order_purchase_timestamp").alias("first_order_date"),
    max("order_purchase_timestamp").alias("last_order_date")
).show()

+-------------------+-------------------+
|   first_order_date|    last_order_date|
+-------------------+-------------------+
|2016-09-04 21:15:19|2018-10-17 17:30:18|
+-------------------+-------------------+



In [57]:
# Converting flag column into a numeric indicator (1/0) before summing

gold_orders = gold_orders.withColumn("approval_delivery_valid_flag", when(col("approval_delivery_valid") == "valid", 1).otherwise("0"))
gold_orders.show(5)

+--------------------+-----------+-------------------+---------------------+--------------------+-------------------+--------------+------------------------+------------+------------------------+-----------------------------+-----------------------+-------------+------------------+---------------------+------------------+----------------------------+
|            order_id|total_price|total_payment_value|total_items_per_order|         customer_id|      customer_city|customer_state|customer_zip_code_prefix|order_status|order_purchase_timestamp|order_delivered_customer_date|approval_delivery_valid|delivery_days|total_installments|payment_mismatch_flag|payment_difference|approval_delivery_valid_flag|
+--------------------+-----------+-------------------+---------------------+--------------------+-------------------+--------------+------------------------+------------+------------------------+-----------------------------+-----------------------+-------------+------------------+------------

In [58]:
# Delivery Performance Metrics

gold_delivery_metrics = gold_orders.groupBy("customer_state").agg(round((avg("delivery_days")),2).alias("avg_delivery_days"),
                                                                 sum("approval_delivery_valid_flag").alias("valid_deliveries"),
                                                                 count("approval_delivery_valid").alias("total_deliveries"))
gold_delivery_metrics.show()

+--------------+-----------------+----------------+----------------+
|customer_state|avg_delivery_days|valid_deliveries|total_deliveries|
+--------------+-----------------+----------------+----------------+
|            SC|            14.91|            3547|            3637|
|            RO|            19.28|             243|             253|
|            PI|            19.39|             476|             495|
|            AM|            26.36|             145|             148|
|            RR|            29.34|              41|              46|
|            GO|            15.54|            1957|            2020|
|            TO|             17.6|             274|             280|
|            MT|             18.0|             886|             907|
|            SP|              8.7|           40437|           41746|
|            ES|            15.72|            1995|            2033|
|            PB|            20.39|             517|             536|
|            RS|            15.25|

In [59]:
# Payment Analysis

gold_payment_analysis = gold_orders.groupBy("total_installments").agg(
    count("order_id").alias("orders"),
    round((sum("total_payment_value")),2).alias("payment_volume")).orderBy(col("total_installments"), ascending = False)
gold_payment_analysis.show()

+------------------+------+--------------+
|total_installments|orders|payment_volume|
+------------------+------+--------------+
|                29|     1|        457.99|
|                26|     1|         62.68|
|                25|     2|       1087.02|
|                24|    16|      10046.91|
|                23|     1|        236.48|
|                22|     2|        269.56|
|                21|     5|       1271.03|
|                20|    29|      23453.33|
|                19|     3|        709.67|
|                18|    28|      14782.53|
|                17|     9|        1534.3|
|                16|    26|      21465.84|
|                15|    80|      35633.76|
|                14|    23|       5776.86|
|                13|    29|       5238.81|
|                12|   146|      47288.56|
|                11|   129|      55758.17|
|                10|  5224|    2162259.22|
|                 9|   693|      147914.7|
|                 8|  4239|    1301031.93|
+----------

In [60]:
gold_orders.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/gold/gold_orders")
gold_customer_metrics.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/gold/gold_customer_metrics")
gold_sale_by_states.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/gold/gold_sale_by_states")
gold_daily_sales.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/gold/gold_daily_sales")
gold_delivery_metrics.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/gold/gold_delivery_metrics")
gold_payment_analysis.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/gold/gold_payment_analysis")